# Data Load

The objective of this notebook is to prepare the working datasets utilizing the datautils to pipeline format data pipeline.

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pprint
from typing import Literal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
project_pwd = Path().cwd().parents[0]
project_pwd = os.path.abspath(project_pwd)
if project_pwd not in sys.path:
    sys.path.append(project_pwd)

sys.dont_write_bytecode = True

In [4]:
from scripts.intro.model import *
from scripts.intro.loader import *
from scripts.intro.block import *
from scripts.intro.activation import *
from scripts.intro.replacement import *
from scripts.intro.eval import *
from scripts.intro.metrics import *

from scripts.data.datautils import *

from scripts.utils import *
from configs.intro_config import *

In [5]:
mcfg = MCFG()
torch.manual_seed(mcfg.seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

## Data Flow

The purpose of this notebook is to document the data preparation part of the project, we will utilize the provided datautils.py as sampling backend. We convert
the datautils format to our pipeline format.

### Datautils.py

training data (C4)

In [6]:
tokenizer = get_tokenizer(mcfg.model_id)
train_samples, val_stream = get_c4(100, mcfg.seed, seqlen=5, model=mcfg.model_id, tokenizer=tokenizer)

Token indices sequence length is longer than the specified maximum sequence length for this model (554556 > 8192). Running this sequence through the model will result in indexing errors


In [7]:
tokenizer

GPT2Tokenizer(name_or_path='HuggingFaceTB/SmolLM2-1.7B', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<filename>", rstrip=False, lstrip=False, single_w

100x shape [1, S]

In [8]:
len(train_samples), train_samples[0][0].shape

(100, torch.Size([1, 5]))

eval stream shape

In [9]:
val_stream.input_ids.shape

torch.Size([1, 1280])

validation data (Wikitext-2)

In [10]:
eval_samples, val_stream = get_wikitext2(1, mcfg.seed, seqlen=5, model=mcfg.model_id, tokenizer=tokenizer)

In [11]:
eval_samples

[(tensor([[645, 384, 436, 216,  34]]),
  tensor([[-100, -100, -100, -100,   34]]))]

In [12]:
val_stream.input_ids.shape

torch.Size([1, 304986])

### ETL pipeline

Datautils to model format conversion

In [13]:
B = mcfg.batch_size

In [14]:
sequences = sequences_from_samples(train_samples[:B])

In [15]:
sequences

[tensor([  339,  2606, 31778, 16687,    17]),
 tensor([  346, 10606,   417,   100,  6654])]

In [16]:
ds = TokenSequencesDataset(sequences)

In [17]:
ds[0]

{'input_ids': tensor([  339,  2606, 31778, 16687,    17]),
 'attention_mask': tensor([1, 1, 1, 1, 1])}

[1, S] -> [S]

In [21]:
ds[0]['input_ids'].shape

torch.Size([5])

loaders - calibration

In [23]:
calib_loader = make_calib_loader(tokenizer, mcfg, dataset="C4")
batch = next(iter(calib_loader))

In [30]:
batch['input_ids']

tensor([[ 3050, 30630,    28,  2853, 13256,    28, 35383, 23096,    28,   281,
          1062, 16452,  5890,  1456, 45953,    30,   378,  6142, 33113,  2254,
           359, 12150,   288,  4031,    30,   216,    36,    32,    32,  5027,
           426,   281,  2267,   260,  6410,   299,   352,  1267,   355,   260,
          1369, 38778,  2581,  3276,  6784,   357,   288,   260, 30525,    30,
         37659,   990,   260,  6142, 33113,  1376,   314,  4035,    28,  4433,
         18978,   359,  4035,   411,   253,   325, 12526,   527,  2824,   354,
           908,  4070,   282,  4433, 18978,   690,   260,  1466,   282,   260,
          3380,  6142, 33113,  2254,    28,   527,  2136,  2223,   260, 18978,
           618,   260,  1376,  1092,   357,  1759,   581,    30,   378, 18978,
           919,   260, 23465, 16748,   418,  3163,   327,  1123, 15230,    30,
           533,  3091, 16947,   715,   347,   653,  2412,  1535,   281, 18385,
          7162,    28,   260,  6142, 33113, 23465,  

In [35]:
batch['input_ids'].shape

torch.Size([2, 128])

In [37]:
[mcfg.batch_size, mcfg.seq_len]

[2, 128]

loaders - validation

In [24]:
eval_loader = make_eval_loader(tokenizer, mcfg, dataset="Wikitext2")
batch2 = next(iter(eval_loader))

In [26]:
batch2

{'input_ids': tensor([[ 1116,   446,  6356,   389,  9226,   352,   446, 38028, 16506,  6356,
            389,  9226,   352,   314,   354,  2321,  4771,  3297,  8552,   284,
          16984, 17523,  1673,   909,   761,   253, 16204,  3394,    29,    48,
          45608,  1791,   335,   260,  8552,  3086,   378,  8593,   281,   216,
             34,    32,    32,    32,  1673,   669,   436,  3922,   411,   253,
          45608,  1791,   281,   260,  1238,  3488,   707,  2855,   411, 15399,
           5970,  3139,  3297,   527,   436,  4940,   281,   216,    34,    32,
             32,    33,   418,   260,  7598,  5620, 22660,  1673,   909,   761,
            253, 16204,  1791,   281,   260,  8552,  3086, 22083,  2322,  1413,
            277,   281,   216,    34,    32,    32,    34,  1673,   533,   216,
             34,    32,    32,    36,   389,  9226,   352, 16176,   253,  1791,
            347,   476, 26984,   476,   281,   260, 12592,   476, 25617, 13285,
            637,    99, 101